# Phase 0 — Lesson 01: Dev Environment
> Your tools shape your thinking. Set them up once, set them up right.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 01
**Type:** Build | **Languages:** Python, Node.js, Rust | **Time:** ~45 minutes
**Prerequisites:** None

This notebook explains *why* each tool exists, not just *how* to install it. Run the code cells to see visualizations that clarify the concepts.


# Why Environment Setup Matters

---


You're about to learn AI engineering across 500+ lessons using Python, TypeScript, Rust, and Julia.
If your environment is broken, every single lesson becomes a fight against tooling instead of learning.

Most people skip environment setup or rush through it. Then they spend hours debugging:
- `ModuleNotFoundError: No module named 'torch'`
- Version conflicts between packages
- Missing CUDA drivers that prevent GPU access

We're going to do this once, properly, so you never have to debug installation during a lesson again.


# Visual: Time Spent on Tooling vs Learning

---


In [ ]:
import matplotlib.pyplot as plt

scenarios = ['Rushed setup', 'Proper setup once']
tooling_hours = [8, 1]
learning_hours = [2, 9]

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(scenarios))
ax.bar(x, tooling_hours, label='Debugging tooling', color='#e74c3c', alpha=0.8)
ax.bar(x, learning_hours, bottom=tooling_hours, label='Actual learning', color='#27ae60', alpha=0.8)

ax.set_ylabel('Hours', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_title('Where Your Time Goes: Rushed vs Proper Setup', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, (t, l) in enumerate(zip(tooling_hours, learning_hours)):
    ax.text(i, t + 0.2, f'{t}h', ha='center', fontweight='bold', color='#e74c3c')
    ax.text(i, t + l + 0.2, f'{l}h', ha='center', fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()


> **The point:** Spending 1 hour on proper setup now saves 7+ hours of debugging later.


# The Four-Layer Environment Stack

---


An AI engineering environment has four layers. Think of it like building a house:

| Layer | Analogy | Components | What breaks if missing |
|-------|---------|------------|------------------------|
| 1. System Foundation | The land and utilities | OS, shell, git, editor, GPU drivers | Nothing works at all |
| 2. Package Managers | The delivery trucks | uv, pnpm, cargo, juliaup | You can't get materials |
| 3. Language Runtimes | The construction workers | Python 3.11+, Node 20+, Rust, Julia | No one can build |
| 4. AI/ML Libraries | The building materials | PyTorch, JAX, transformers | Nothing useful gets built |

**We install bottom-up.** Each layer depends on the one below it. If you try to install PyTorch (layer 4) before you have Python (layer 3), it fails.


# Visual: What Breaks When You Skip Layers

---


In [ ]:
import matplotlib.pyplot as plt

layers = [
    'Layer 4: AI Libraries (PyTorch)',
    'Layer 3: Language Runtimes (Python)',
    'Layer 2: Package Managers (uv)',
    'Layer 1: System Foundation (OS + Drivers)',
]

# Functionality % if THAT specific layer is missing
impact = [100, 100, 100, 0]  # only layer 1 being missing leaves 0%

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60']
bars = ax.barh(layers, impact, color=colors, alpha=0.8)
ax.set_xlabel('Functionality remaining if this layer is missing (%)', fontsize=10)
ax.set_title('Impact of Missing Each Layer', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.set_xlim(0, 110)

for i, (bar, val) in enumerate(zip(bars, impact)):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()


> **Key insight:** If Layer 1 (System Foundation) is missing, **everything above it is 0% functional.**
That's why we check OS, shell, git, and GPU drivers first.


# Step 1: System Foundation — The Land and Utilities

---


Before any programming language, you need the basics on your computer:

**macOS:**
```bash
xcode-select --install
brew install git curl wget
```

**Ubuntu/Debian (including WSL2):**
```bash
sudo apt update && sudo apt install -y build-essential git curl wget unzip
```

**Windows:** Use WSL2 (Windows Subsystem for Linux)
```bash
wsl --install -d Ubuntu-24.04
```

Why these specific tools:
- **git:** Version control (you'll use it in Lesson 02)
- **curl / wget:** Download files from the internet
- **unzip:** Extract compressed archives
- **build-essential:** Compilers needed to install some Python packages


# Step 2: Python with uv — Why Not Just pip?

---


Python comes with `pip`, but `pip` has a problem: it resolves dependencies **one at a time**, sequentially. For AI projects with large packages like PyTorch, this can take minutes.

**uv** is written in Rust and resolves all dependencies **in parallel**. It's typically 10-100x faster.

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh

uv python install 3.12

uv venv
source .venv/bin/activate  # or .venv\\Scripts\\activate on Windows

uv pip install numpy matplotlib jupyter
```

**What each command does:**
- `uv python install 3.12` — Downloads and installs Python 3.12
- `uv venv` — Creates a virtual environment (isolated folder for this project's packages)
- `source .venv/bin/activate` — Activates that environment so your terminal uses it
- `uv pip install numpy matplotlib jupyter` — Installs packages into that environment


# Visual: Install Time — uv vs pip

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

packages = ['numpy', 'pandas', 'matplotlib', 'torch', 'scikit-learn']
pip_times = [8.2, 12.5, 6.1, 45.3, 9.8]
uv_times = [0.9, 1.4, 0.7, 5.2, 1.1]

x = np.arange(len(packages))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, pip_times, width, label='pip', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, uv_times, width, label='uv', color='#27ae60', alpha=0.8)

ax.set_xlabel("Python Package")
ax.set_ylabel("Install Time (seconds, log scale)")
ax.set_title("uv vs pip: Install Time for Common AI Packages", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(packages)
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for i, (p, u) in enumerate(zip(pip_times, uv_times)):
    speedup = round(p / u, 1)
    ax.text(i, max(p, u) * 1.5, f"{speedup}x", ha="center", fontweight="bold", fontsize=10,
            color="#27ae60")

plt.tight_layout()
plt.show()


> **PyTorch takes 45 seconds with pip, 5 seconds with uv.** Across a whole course, that's hours saved.


# Virtual Environments: Why Not Just Install Everything Globally?

---


Imagine two projects:
- **Project A** needs PyTorch 1.13
- **Project B** needs PyTorch 2.1

If you install both globally, they conflict. One breaks.

A **virtual environment** is like a separate apartment for each project — it has its own Python, its own packages, and its own versions. Nothing leaks between them.

This is why we run `uv venv` before installing anything.


# Step 3: Node.js with pnpm — For TypeScript Lessons

---


Phases 13-17 cover agents, MCP servers, and web apps using TypeScript. You need Node.js for that.

**fnm** (Fast Node Manager) lets you install and switch Node versions easily.

```bash
curl -fsSL https://fnm.vercel.app/install | bash
fnm install 22
fnm use 22

npm install -g pnpm

node -e "console.log('Node', process.version)"
```

**pnpm** is Node's package manager (like `uv` is for Python). It's fast and disk-efficient.


# Step 4: Rust — For Performance-Critical Code

---


Some lessons (inference optimization, systems programming) need Rust for speed.

```bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh

rustc --version
cargo --version
```

**cargo** is Rust's package manager and build tool (like `uv` for Python or `pnpm` for Node).


# Step 5: Julia — Optional, for Math-Heavy Lessons

---


Julia is excellent for mathematical computing. It's optional — you only need it if a specific lesson requires it.

```bash
curl -fsSL https://install.julialang.org | sh

julia -e 'println("Julia ", VERSION)'
```


# Step 6: GPU Setup — CUDA vs MPS vs CPU

---


AI training does massive matrix math. A GPU can be 10-100x faster than a CPU for this.

**NVIDIA (Linux / Windows):** Uses CUDA
```bash
nvidia-smi

# Install PyTorch with CUDA support
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```

**macOS / Apple Silicon (M1/M2/M3/M4):** No CUDA on Mac. Use MPS (Metal Performance Shaders) instead.
```bash
# Plain install — includes MPS backend automatically
uv pip install torch torchvision torchaudio
```

**No GPU?** No problem. Most lessons work on CPU. For heavy training, use Google Colab or cloud GPUs.


# Visual: GPU Backend Options by Platform

---


In [ ]:
import matplotlib.pyplot as plt

platforms = ['Linux + NVIDIA GPU', 'macOS + Apple Silicon', 'CPU Only (any computer)']
backends = ['CUDA', 'MPS (Metal)', 'None']
speedup = ['20-100x faster', '5-20x faster', 'Baseline (1x)']
best_for = ['Training large models', 'Development on Mac', 'Learning, inference']

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = [[p, b, s, f] for p, b, s, f in zip(platforms, backends, speedup, best_for)]
table = ax.table(
    cellText=table_data,
    colLabels=['Platform', 'GPU Backend', 'Speed vs CPU', 'Best For'],
    loc='center',
    cellLoc='left',
    colColours=['#2c3e50', '#2c3e50', '#2c3e50', '#2c3e50']
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

for i in range(4):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

plt.title("GPU Backend Options", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Verify your setup works:**
```python
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {torch.backends.mps.is_available()}")
```
- On NVIDIA: CUDA = True
- On Apple Silicon: MPS = True
- On plain CPU: Both = False (still works!)


# Step 7: Verify Your Environment

---


Run this from the **repository root** (the folder containing `README.md` and `phases/`):

```bash
python3 phases/00-setup-and-tooling/01-dev-environment/code/verify.py --route beginner
```

This checks only what you need to start. It won't fail you for missing tools used in later lessons.
Add `--show-later` if you want to see optional requirements too.

When it passes, you'll see:
```text
Ready to start Beginner course.
Next: python3 phases/01-math-foundations/01-linear-algebra-intuition/code/vectors.py
```


# Visual: Which Language When?

---


In [ ]:
import matplotlib.pyplot as plt

phases = ['Phases 1-12', 'Phases 13-17', 'Phases 12, 15-17', 'Phase 1']
languages = ['Python', 'TypeScript', 'Rust', 'Julia']
package_managers = ['uv', 'pnpm', 'cargo', 'Pkg']
purposes = [
    'ML, DL, NLP, Vision, Audio, LLMs',
    'Tools, Agents, Swarms, Infra',
    'Performance-critical systems',
    'Math foundations'
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = list(zip(phases, languages, package_managers, purposes))
table = ax.table(
    cellText=table_data,
    colLabels=['Course Phases', 'Language', 'Package Manager', 'Used For'],
    loc='center',
    cellLoc='left',
    colColours=['#2c3e50', '#2c3e50', '#2c3e50', '#2c3e50']
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

for i in range(4):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

plt.title("Languages and Package Managers Across the Curriculum", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


| Language | Phases | Package Manager | Used For |
|----------|--------|-----------------|----------|
| Python | 1-12 | uv | ML, DL, NLP, Vision, Audio, LLMs |
| TypeScript | 13-17 | pnpm | Tools, Agents, Swarms, Infra |
| Rust | 12, 15-17 | cargo | Performance-critical systems |
| Julia | Phase 1 | Pkg | Math foundations |


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** Why do AI projects need a separate virtual environment?
> **Explanation:** Without isolation, Project A needing PyTorch 1.13 and Project B needing PyTorch 2.1 would conflict. A virtual environment gives each project its own private copy of packages.

**Q2:** What does CUDA provide for AI workloads?
> **Explanation:** CUDA lets PyTorch run matrix operations on thousands of NVIDIA GPU cores simultaneously, making training 10-100x faster than CPU.

**Q3:** In the four-layer stack, which layer installs first?
> **Explanation:** Layer 1 — System Foundation (OS, shell, GPU drivers). You can't install Python without an OS.

**Q4:** What is uv?
> **Explanation:** A Rust-based Python package installer that resolves dependencies in parallel — 10-100x faster than pip.

**Q5:** How do you verify PyTorch GPU access?
> **Explanation:** `import torch; print(torch.cuda.is_available())`. On Mac, use `torch.backends.mps.is_available()`.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Run the verification script.
```bash
python3 phases/00-setup-and-tooling/01-dev-environment/code/verify.py --route beginner
```

**Exercise 2:** Create a virtual environment and install PyTorch.
```bash
uv venv
source .venv/bin/activate
uv pip install torch torchvision torchaudio
```

**Exercise 3:** Write 'hello world' in each language.


In [ ]:
# Python
print("Hello from Python!")


**Rust:**
```rust
fn main() {
    println!("Hello from Rust!");
}
```

**Node.js:**
```javascript
console.log("Hello from Node.js!");
```

**Julia:**
```julia
println("Hello from Julia!")
```


# Key Terms

---


| Term | What it means |
|------|---------------|
| Virtual Environment | Isolated folder with its own Python and packages |
| uv | Fast Python package installer (Rust-based, parallel resolution) |
| fnm | Fast Node.js version manager |
| cargo | Rust's package manager and build tool |
| CUDA | NVIDIA's GPU computing platform for parallel math |
| MPS | Apple's Metal Performance Shaders (GPU on macOS) |
| Repository root | Folder containing README.md and phases/ |


# Next Steps

---


When verification passes, you're ready for Lesson 02: Git and Collaboration.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
